# Hello World Deep Learning in Medical Imaging

A short demonstration of the `hello_world_medical_imaging` package, which now holds the code of the tutorial by Lakhani et al. ([article](https://doi.org/10.1007/s10278-018-0079-6)). The notebook as it was published is at the Git tag `step-0-notebook`; the cells it had are now functions in `src/hello_world_medical_imaging/`. The same pipeline runs from the command line with `uv run hello-world-medical-imaging`.

In [ ]:
# Parameters: papermill can override them (papermill ... -p config_file my-experiment.toml)
config_file = "config.toml"

In [ ]:
import os
from pathlib import Path

import keras
from IPython.display import Image

from hello_world_medical_imaging.config import load_config
from hello_world_medical_imaging.data import (
    TEST_SUBDIRECTORY,
    download_dataset,
    load_image_datasets,
    synthetic_datasets,
)
from hello_world_medical_imaging.model import build_model
from hello_world_medical_imaging.predict import label_from_score, predict_image
from hello_world_medical_imaging.training import plot_history, train

# Paths in config.toml are relative to the root of the repository, as for the command line
if Path.cwd().name == "notebooks":
    os.chdir("..")

config = load_config(Path(config_file))
keras.utils.set_random_seed(config.seed)

## Training on the radiographs

Each step is one call: download the images, read them, build the model, train it.

In [ ]:
dataset_directory = download_dataset(config.dataset_url, config.dataset_directory)
train_dataset, validation_dataset = load_image_datasets(
    dataset_directory, config.image_size, config.batch_size, config.seed
)
model = build_model(config.image_size, config.learning_rate, config.augmentation)
history = train(model, train_dataset, validation_dataset, config.epochs)
Image(filename=plot_history(history, config.output_directory / "training_curves.png"))

In [ ]:
for image_path in sorted((dataset_directory / TEST_SUBDIRECTORY).glob("*.png")):
    score = predict_image(model, image_path, config.image_size)
    print(f"{image_path.name}: score {score:.3f}, predicted {label_from_score(score)}")

## The same training on synthetic images

`train` receives its data as arguments, so another source can replace the radiographs without any change to the model or the training code. Here, noise images with random labels and a model with random weights, at 75 pixels: the pipeline runs in seconds and learns nothing, which is all a test needs.

In [ ]:
noise_train, noise_validation = synthetic_datasets(image_size=75, batch_size=4, seed=config.seed)
small_model = build_model(75, config.learning_rate, pretrained=False)
noise_history = train(small_model, noise_train, noise_validation, epochs=1)